# The Gymnasium interface of a planning problem

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/teaching/reinforcement_learning/gymnasium_interface.ipynb)

[Gymnasium](https://gymnasium.farama.org) is the standard Python interface for reinforcement-learning environments. Almost every RL library trains against it, so knowing its contract is knowing how to plug any task into any of them. The contract is small: two spaces and two methods.

This notebook reads that contract in optimal-control terms on a pendulum swing-up. It writes an environment by hand, checks it with Gymnasium's own validator, obtains the same environment from minilink, measures what stepping it from Python costs, and trains a policy through it. The native planner of [`11_reinforcement_learning`](../../tutorial/11_reinforcement_learning.ipynb) runs this same step as a compiled function; the mathematics of the algorithms is in [`policy_gradient_to_ppo`](policy_gradient_to_ppo.ipynb).

In [ ]:
# Local: minilink already installed. Colab: clone + path + RL libraries.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q gymnasium stable-baselines3")

## 1. The task

A torque-limited pendulum ($\theta = 0$ hanging, $\theta = \pi$ upright) with less torque than gravity, so reaching the top takes a pumping motion. The cost is $J = \int g\,dt$ with a running cost zero upright and two hanging, and the start is drawn uniformly over the whole circle with small rates. The `StochasticPlanningProblem` states all of it once.

In [ ]:
import importlib.util
import time

import gymnasium as gym
import jax.numpy as jnp
import numpy as np
from gymnasium import spaces
from gymnasium.utils.env_checker import check_env

from minilink import CostFunction, Pendulum
from minilink.control import angle_features
from minilink.interfaces.gymnasium import SB3Controller, Sys2Gym
from minilink.planning import (
    MonteCarloEvaluator,
    ReinforcementLearningPlanner,
    StochasticPlanningProblem,
    Uniform,
)

TORQUE = 4.0  # Nm, below m g l = 9.81 Nm
DT = 0.05  # control period: one environment step

plant = Pendulum()
plant.inputs["u"].lower_bound = np.array([-TORQUE])
plant.inputs["u"].upper_bound = np.array([TORQUE])
plant.state.lower_bound = np.array([-4 * np.pi, -20.0])  # the allowed box
plant.state.upper_bound = np.array([4 * np.pi, 20.0])


class SwingUpCost(CostFunction):
    def g(self, x, u, t=0.0, params=None):
        theta, dtheta = x
        return (1.0 + jnp.cos(theta)) + 0.01 * dtheta**2 + 0.01 * u[0] ** 2

    def h(self, x, t=0.0, params=None):
        return 0.0


cost = SwingUpCost()
problem = StochasticPlanningProblem(
    plant,
    cost=cost,
    tf=np.inf,
    x0_distribution=Uniform([-np.pi, -1.0], [np.pi, 1.0]),
)

## 2. The contract: spaces, reset, step

| Gymnasium | Optimal control | Here |
| --- | --- | --- |
| `observation_space` | the set of observations $y$ | the state box |
| `action_space` | the admissible inputs $U$ | $\lvert u \rvert \le 4$ Nm |
| `reset(seed)` returns `(y, info)` | draw $x_0$ from the start distribution | uniform on the circle |
| `step(u)` returns `(y, r, terminated, truncated, info)` | one control period of the task | see below |

One `step` is four sub-steps:

1. **Dynamics:** $x_{k+1} = f(x_k, u_k)$ over $\Delta t$, the input held.
2. **Reward:** $r_k = -g(x_k, u_k)\,\Delta t$. Gymnasium maximizes reward, so the sign flips the cost.
3. **Observation:** $y_{k+1} = h(x_{k+1})$, the full state for this plant.
4. **Episode end:** *terminated* when nothing is left to count (a goal, a crash, the end of a finite horizon after its terminal cost), *truncated* when the episode is cut short but the future still has value (a time limit, or leaving the box without a price). A learning algorithm bootstraps a truncated episode with the value of its last state and not a terminated one, so the distinction changes what is learned.

## 3. An environment written by hand

The class holds the state between calls, which is what makes an environment an *object*: `reset` sets it, `step` advances it. Each block of `step` is one of the four sub-steps above. The dynamics come from the catalog plant's `f`, integrated by one explicit Euler step.

In [ ]:
class PendulumSwingUpEnv(gym.Env):
    """The swing-up task written against the Gymnasium interface by hand."""

    def __init__(self, dt=DT, tf=10.0):
        self.dt, self.tf = dt, tf
        x_lb, x_ub = plant.state.lower_bound, plant.state.upper_bound
        u_lb, u_ub = plant.inputs["u"].lower_bound, plant.inputs["u"].upper_bound
        self.observation_space = spaces.Box(x_lb.astype(np.float32), x_ub.astype(np.float32))
        self.action_space = spaces.Box(u_lb.astype(np.float32), u_ub.astype(np.float32))

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # Initial state: a draw from the start distribution
        self.x = self.np_random.uniform([-np.pi, -1.0], [np.pi, 1.0])
        self.t = 0.0
        return self.x.astype(np.float32), {}

    def step(self, u):
        x, t, dt = self.x, self.t, self.dt
        u = np.clip(np.asarray(u, dtype=float), self.action_space.low, self.action_space.high)

        # 1. Dynamics: x' = x + f(x, u) dt, one Euler step with the input held
        x_next = x + plant.f(x, u, t) * dt

        # 2. Reward: r = -g(x, u) dt
        reward = -float(cost.g(x, u, t)) * dt

        # 3. Observation: y = h(x'), the full state
        y = x_next.astype(np.float32)

        # 4. Episode end: this task never terminates; the time limit and leaving the box truncate
        terminated = False
        outside = np.any(x_next < self.observation_space.low) or np.any(
            x_next > self.observation_space.high
        )
        truncated = bool(t + dt > self.tf or outside)

        self.x, self.t = x_next, t + dt
        return y, reward, terminated, truncated, {}

Gymnasium ships a validator of the contract. It passes, with two recommendations: it cannot test render modes this environment does not offer, and it suggests a symmetric, normalized action space, which is why minilink's neural law outputs a normalized action and maps it onto the port bounds.

A random policy then runs one episode through the interface. Its return is minus its cost.

In [ ]:
env = PendulumSwingUpEnv()
check_env(env)

y, info = env.reset(seed=0)
episode_return, k = 0.0, 0
terminated = truncated = False
while not (terminated or truncated):
    y, r, terminated, truncated, info = env.step(env.action_space.sample())
    episode_return += r
    k += 1
print(f"random policy: {k} steps, return {episode_return:.2f}, so J = {-episode_return:.2f}")

## 4. The same environment from minilink

`Sys2Gym.from_problem` builds the environment from the problem instead of by hand: `reset` draws from the problem's start distribution, the reward is its running cost, and its exit and horizon rules decide *terminated* or *truncated*. It integrates one RK4 step on the compiled plant; with `integrator="euler"` it is the class above, step for step.

In [ ]:
gym_env = Sys2Gym.from_problem(problem, dt=DT, integrator="euler")
check_env(gym_env)
print("observation space:", gym_env.observation_space)
print("action space:     ", gym_env.action_space)

# Same state, same actions: the two environments agree to rounding
env.reset(seed=1)
gym_env.reset(seed=1)
env.x = gym_env.x = np.array([0.3, 0.0])
largest = 0.0
for k in range(100):
    u = np.array([TORQUE * np.sin(0.3 * k)])
    _, r_hand, *_ = env.step(u)
    _, r_lib, *_ = gym_env.step(u)
    largest = max(largest, float(np.max(np.abs(env.x - gym_env.x))), abs(r_hand - r_lib))
print("largest difference over 100 steps:", largest)

## 5. What a Python loop costs

Gymnasium steps one plant at a time, from Python. The native planner runs the same step as a pure JAX function over many plants inside one compiled loop, so it can *train*, collecting experience and updating the policy together, faster than a Gymnasium loop merely *steps* with a random policy.

In [ ]:
rk4_env = Sys2Gym.from_problem(problem, dt=DT)
rk4_env.reset(seed=2)
t0 = time.time()
for _ in range(5000):
    _, _, terminated, truncated, _ = rk4_env.step(rk4_env.action_space.sample())
    if terminated or truncated:
        rk4_env.reset()
loop_rate = 5000 / (time.time() - t0)

planner = ReinforcementLearningPlanner(
    problem,
    dt=DT,
    features=angle_features(angles=[0], scales={1: 0.1}),
    hidden=(32, 32),
    n_envs=64,
    n_steps=32,
    batch_size=256,
    learning_rate=3e-3,
    gamma=0.97,
    verbose=0,
)
planner.solve(timesteps=120_000)
train_rate = float(np.median([record["fps"] for record in planner.history[1:]]))
rl_ctl = planner.get_controller()

print(f"Gymnasium loop, stepping only: {loop_rate:8.0f} steps/s")
print(f"native planner, training:      {train_rate:8.0f} steps/s")

## 6. Training through the interface

Any library that speaks Gymnasium trains on these environments. Stable-Baselines3 is the usual reference; its trained model comes back into minilink as a controller block through `SB3Controller`, and scores on the same Monte Carlo yardstick as the native law. The cell installs nothing: without the package it says so, and the comparison below keeps only the native law.

The Gymnasium policy observes the raw state here, while the native law used periodic angle features, so the comparison is between two setups rather than two libraries.

In [ ]:
sb3_ctl = None
if importlib.util.find_spec("stable_baselines3") is None:
    print("Stable-Baselines3 is not installed: pip install stable-baselines3, then re-run this cell.")
else:
    from stable_baselines3 import PPO

    model = PPO("MlpPolicy", rk4_env, gamma=0.97, verbose=0)
    t0 = time.time()
    model.learn(total_timesteps=120_000)
    print(f"Stable-Baselines3 PPO: 120000 steps in {time.time() - t0:.0f} s")
    sb3_ctl = SB3Controller(model, sys=plant)

In [ ]:
evaluator = MonteCarloEvaluator(
    problem, dt=DT, n_trials=100, episode_length=10.0, backend="numpy", seed=1
)
laws = {"native planner": rl_ctl}
if sb3_ctl is not None:
    laws["Stable-Baselines3"] = sb3_ctl
for name, law in laws.items():
    print(f"{name:18s} |", evaluator.evaluate(law))

## What to read next

- [`11_reinforcement_learning`](../../tutorial/11_reinforcement_learning.ipynb): the planner API — the problem, the rollout environment, the discount, both algorithm families, the learned law as a block.
- [`policy_gradient_to_ppo`](policy_gradient_to_ppo.ipynb): what PPO does with the transitions an environment returns.
- `minilink/interfaces/gymnasium.py`: `Sys2Gym`, `ProblemEnv` and `SB3Controller`, the bridge used above.